# Thesis Progress Summary

This notebook gives a short overview of the current thesis progress:
- datasets used
- preprocessing and multimodal merging
- next possible steps

## 1. Datasets

The thesis uses:
- PaySim for tabular fraud detection
- Document image datasets for image-based fraud detection
- A synthetic merged multimodal dataset for multimodal experiments


In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

paysim_train = pd.read_csv(processed_dir / "paysim_train.csv")
paysim_val = pd.read_csv(processed_dir / "paysim_val.csv")
paysim_test = pd.read_csv(processed_dir / "paysim_test.csv")

image_train = pd.read_csv(processed_dir / "image_train.csv")
image_val = pd.read_csv(processed_dir / "image_val.csv")
image_test = pd.read_csv(processed_dir / "image_test.csv")

mm_train = pd.read_csv(processed_dir / "mm_train.csv")
mm_val = pd.read_csv(processed_dir / "mm_val.csv")
mm_test = pd.read_csv(processed_dir / "mm_test.csv")

print("PaySim:")
print(paysim_train.shape, paysim_val.shape, paysim_test.shape)

print("\nImage:")
print(image_train.shape, image_val.shape, image_test.shape)

print("\nMultimodal:")
print(mm_train.shape, mm_val.shape, mm_test.shape)

PaySim:
(4453834, 12) (636262, 12) (1272524, 12)

Image:
(6653, 10) (1743, 10) (2425, 10)

Multimodal:
(6653, 29) (1658, 29) (2425, 29)


In [4]:
summary = pd.DataFrame({
    "Dataset": ["PaySim", "Image", "Multimodal"],
    "Train": [len(paysim_train), len(image_train), len(mm_train)],
    "Validation": [len(paysim_val), len(image_val), len(mm_val)],
    "Test": [len(paysim_test), len(image_test), len(mm_test)]
})

summary

,Dataset,Train,Validation,Test
0,PaySim,4453834,636262,1272524
1,Image,6653,1743,2425
2,Multimodal,6653,1658,2425


## 2. Label distributions


In [ ]:
def show_distribution(name, df, target_col):
    print(f"\n{name}")
    print(df[target_col].value_counts())
    print(df[target_col].value_counts(normalize=True) * 100)

show_distribution("PaySim train", paysim_train, "isFraud")
show_distribution("Image train", image_train, "original_label")  
show_distribution("Multimodal train", mm_train, "final_label")


PaySim train
isFraud
0    4448085
1       5749
Name: count, dtype: int64
isFraud
0    99.87092
1     0.12908
Name: proportion, dtype: float64

Image train
original_label
0           2796
forged      2012
attack      1230
bonafide     615
Name: count, dtype: int64
original_label
0           42.026154
forged      30.241996
attack      18.487900
bonafide     9.243950
Name: proportion, dtype: float64

Multimodal train
final_label
0    3411
1    3242
Name: count, dtype: int64
final_label
0    51.270104
1    48.729896
Name: proportion, dtype: float64


## 3. Multimodal merge logic

I first analyzed the tabular and image datasets separately. Then I merged the image datasets into one unified image dataset with binary labels. Finally, I merged this image dataset with PaySim using a synthetic pairing strategy to create the multimodal dataset.

Since the tabular and image datasets do not have natural one-to-one matching samples.

To create the multimodal dataset, I used the following rules:
- samples were matched only within the same split (train, validation, test)
- samples were matched only within the same binary label
- samples were shuffled before pairing
- the final number of multimodal pairs in each split and class was limited by the smaller modality

This created a synthetic but label-aligned multimodal dataset without split leakage.

In [3]:
cols_to_show = [
    "mm_id",
    "split",
    "final_label",
    "tab_step",
    "tab_type",
    "tab_amount",
    "img_image_path",
    "img_image_class",
    "img_source_dataset"
]

mm_train[cols_to_show].head(10)

,mm_id,split,final_label,tab_step,tab_type,tab_amount,img_image_path,img_image_class,img_source_dataset
0,mm_train_1_002875,train,1,343,4,802417.45,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fantasyid
1,mm_train_0_000217,train,0,251,3,40693.51,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,fantasyid
2,mm_train_0_003214,train,0,188,1,279091.56,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
3,mm_train_1_001104,train,1,17,1,1639676.27,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
4,mm_train_0_001544,train,0,252,0,79797.27,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
5,mm_train_0_000247,train,0,349,1,313269.65,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
6,mm_train_1_002906,train,1,702,4,10000000.00,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
7,mm_train_0_000230,train,0,404,3,442.49,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
8,mm_train_1_001470,train,1,215,1,319887.08,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fantasyid
9,mm_train_1_002379,train,1,726,1,1539474.57,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fantasyid


## 4. Experiment setup

Current planned experiments:
1. Tabular-only model on full PaySim
2. Tabular-only model on merged subset
3. Image-only model on full image dataset
4. Image-only model on merged subset
5. Multimodal model on merged subset

## 5. Current observations

- The multimodal dataset was created successfully.
- The merged dataset contains both tabular PaySim features and image paths.
- The multimodal dataset is smaller than the original datasets because pairing is limited by the smaller class size in each split.
- A fair comparison is possible on the merged subset.

## 6. Next steps

The next step is to train and compare:
- full tabular baseline
- merged tabular baseline
- full image baseline
- merged image baseline
- merged multimodal baseline